# Copilot Cost Consumption — Ingester (Fabric)

Lands the **Microsoft 365 Admin Center → Copilot → Cost management** per-user CSV export into a Lakehouse Delta table the dashboard reads for credit attribution. **Auto-detects two export shapes** and maps both to one unified contract:

1. **Surface split** — `User Principal Name`, `Cowork Credits`, `WorkIQ Credits`, `Other Credits`, `Last Activity Date` (the Cowork/WorkIQ/Other view).
2. **Per-user usage** — `Display Name`, `User Principal Name`, `Monthly credit limit`, `Monthly credits used`, `% Used`, `Session Count`, `Microsoft 365 Copilot license`, `Last activity date` (budget / utilization view).

Header matching is **case-insensitive** (handles `Last activity date` vs `Last Activity Date`). Additive to the PPAC `credit_consumption_*` tables — not a replacement.

```
M365 Admin Center  ─(export)▶  Power Automate flow  ▶  Lakehouse Files/cost_consumption/*.csv  ▶  THIS notebook  ▶  Delta
```

**Unified Delta contract** (`dbo.copilot_cost_consumption`): `User_Principal_Name`, `Display_Name`, `Cowork_Credits`, `WorkIQ_Credits`, `Other_Credits`, `Total_Credits`, `Monthly_Credit_Limit`, `Pct_Used`, `Session_Count`, `M365_Copilot_Licensed`, `Last_Activity_Date`, `SourceFile`, `LoadDate`. Columns absent from a given export load as null. `Total_Credits` = sum of surface credits, or `Monthly credits used` for the usage shape.

**Graceful / no permissions** — empty folder writes an empty correctly-named table; reads files only.

## 1. Configuration

Tag this cell as the pipeline **`parameters`** cell so the adoption pipeline can override `WRITE_MODE`.

In [ ]:
# === CONFIG ===  (tag this cell as the pipeline `parameters` cell)

SOURCE_DIR   = 'Files/cost_consumption'   # MUST match TargetFolder in the landing flows
SOURCE_GLOB  = '*'
OUTPUT_TABLE = 'dbo.copilot_cost_consumption'

WRITE_MODE   = 'overwrite'   # snapshot (per-user point-in-time export)
ADD_LINEAGE  = True          # SourceFile + LoadDate
STRICT       = False         # False = empty table on missing export; True = raise

## 2. Helpers

`_canon` renames raw headers to the unified contract **case-insensitively** (so both export shapes and casings map). `_typed` coerces numerics, parses dates (ISO timestamp or US `M/d/yyyy`), derives `Total_Credits` (surface sum, or monthly used), and normalises `Pct_Used` to a 0–1 fraction.

In [ ]:
import os, re, datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType

_LOAD_TS = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
# Spark 3 strict datetime parser throws on unmatched formats; LEGACY returns null instead.
spark.conf.set('spark.sql.legacy.timeParserPolicy', 'LEGACY')

def _local(p): return p if p.startswith('/') else f'/lakehouse/default/{p}'

def _list_matches(pattern):
    rx = re.compile('^' + re.escape(pattern).replace('\\*', '.*') + '$', re.IGNORECASE)
    try:
        entries = notebookutils.fs.ls(SOURCE_DIR)
    except Exception:
        return []
    return [f'{SOURCE_DIR}/{e.name}' for e in entries
            if (not e.isDir) and rx.match(e.name) and e.name.lower().endswith('.csv')]

def _read_csv(path):
    return (spark.read.option('header', True).option('multiLine', True)
            .option('escape', '"').option('encoding', 'UTF-8').csv(path))

# canonical contract column <- candidate source headers (lower-cased, BOM-stripped match)
_SYN = {
    'User_Principal_Name':   ['user principal name', 'userprincipalname', 'upn'],
    'Display_Name':          ['display name'],
    'Cowork_Credits':        ['cowork credits'],
    'WorkIQ_Credits':        ['workiq credits'],
    'Other_Credits':         ['other credits'],
    'Monthly_Credits_Used':  ['monthly credits used'],
    'Monthly_Credit_Limit':  ['monthly credit limit'],
    'Pct_Used':              ['% used', 'percent used'],
    'Session_Count':         ['session count'],
    'M365_Copilot_Licensed': ['microsoft 365 copilot license', 'microsoft 365 copilot licensed'],
    'Last_Activity_Date':    ['last activity date'],
}
def _canon(df):
    lower = {c.lower().strip().replace('\ufeff', ''): c for c in df.columns}
    out = df
    for canon, cands in _SYN.items():
        for cand in cands:
            if cand in lower and lower[cand] != canon:
                out = out.withColumnRenamed(lower[cand], canon); break
    return out

_SURFACE = ['Cowork_Credits', 'WorkIQ_Credits', 'Other_Credits']
def _typed(df):
    cols = set(df.columns); out = df
    for c in _SURFACE + ['Monthly_Credits_Used', 'Monthly_Credit_Limit', 'Pct_Used', 'Session_Count']:
        if c in cols: out = out.withColumn(c, F.col(c).cast('double'))
    if 'Last_Activity_Date' in cols:
        out = out.withColumn('Last_Activity_Date', F.coalesce(
            F.to_date(F.substring(F.col('Last_Activity_Date'), 1, 10), 'yyyy-MM-dd'),
            F.to_date('Last_Activity_Date', 'M/d/yyyy')))
    surf = [c for c in _SURFACE if c in cols]
    if surf:
        tot = None
        for c in surf:
            term = F.coalesce(F.col(c).cast('double'), F.lit(0.0))
            tot = term if tot is None else tot + term
        out = out.withColumn('Total_Credits', tot)
    elif 'Monthly_Credits_Used' in cols:
        out = out.withColumn('Total_Credits', F.col('Monthly_Credits_Used').cast('double'))
    else:
        out = out.withColumn('Total_Credits', F.lit(None).cast('double'))
    if 'Pct_Used' in cols:
        out = out.withColumn('Pct_Used', F.col('Pct_Used').cast('double') / F.lit(100.0))
    return out

print(f'Load timestamp: {_LOAD_TS}')
print(f'Source folder : {SOURCE_DIR}  (exists: {os.path.isdir(_local(SOURCE_DIR))})')

## 3. Ingest → Delta

Globs every `.csv`, canonicalises headers, types, derives `Total_Credits`, then **conforms each file to the full unified schema** (missing columns added as null) before union — so a surface export and a usage export can even be dropped together. Missing/empty folder writes an empty correctly-named table.

In [ ]:
UNIFIED = ['User_Principal_Name','Display_Name','Cowork_Credits','WorkIQ_Credits','Other_Credits',
           'Total_Credits','Monthly_Credit_Limit','Pct_Used','Session_Count','M365_Copilot_Licensed',
           'Last_Activity_Date']
_DTYPE = {'Cowork_Credits':'double','WorkIQ_Credits':'double','Other_Credits':'double',
          'Total_Credits':'double','Monthly_Credit_Limit':'double','Pct_Used':'double',
          'Session_Count':'bigint','Last_Activity_Date':'date'}

def _conform(df):
    for c in UNIFIED:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None).cast(_DTYPE.get(c, 'string')))
    keep = UNIFIED + ([ 'SourceFile','LoadDate'] if ADD_LINEAGE else [])
    return df.select(*[c for c in keep if c in df.columns])

def _empty():
    f = [StructField('User_Principal_Name', StringType()), StructField('Display_Name', StringType()),
         StructField('Cowork_Credits', DoubleType()), StructField('WorkIQ_Credits', DoubleType()),
         StructField('Other_Credits', DoubleType()), StructField('Total_Credits', DoubleType()),
         StructField('Monthly_Credit_Limit', DoubleType()), StructField('Pct_Used', DoubleType()),
         StructField('Session_Count', LongType()), StructField('M365_Copilot_Licensed', StringType()),
         StructField('Last_Activity_Date', DateType())]
    if ADD_LINEAGE: f += [StructField('SourceFile', StringType()), StructField('LoadDate', StringType())]
    return spark.createDataFrame([], StructType(f))

def _write(df, table):
    (df.write.mode(WRITE_MODE).option('overwriteSchema', 'true')
        .option('delta.columnMapping.mode', 'name')
        .option('delta.minReaderVersion', '2').option('delta.minWriterVersion', '5')
        .format('delta').saveAsTable(table))

matches = _list_matches(SOURCE_GLOB)
if not matches:
    msg = f'no .csv matched "{SOURCE_GLOB}" in {SOURCE_DIR}'
    if STRICT: raise FileNotFoundError(f'{OUTPUT_TABLE}: {msg}')
    print(f'⚠  {OUTPUT_TABLE} — {msg}; writing EMPTY table.')
    _write(_empty(), OUTPUT_TABLE); rows = 0
else:
    frames = []
    for path in matches:
        d = _conform(_typed(_canon(_read_csv(path))))
        if ADD_LINEAGE:
            d = d.withColumn('SourceFile', F.lit(os.path.basename(path))).withColumn('LoadDate', F.lit(_LOAD_TS))
        frames.append(d)
    df = frames[0]
    for d in frames[1:]: df = df.unionByName(d, allowMissingColumns=True)
    rows = df.count(); _write(df, OUTPUT_TABLE)
    print(f'✓  {OUTPUT_TABLE} — {len(matches)} file(s), {rows:,} rows -> written ({WRITE_MODE})')

print('\nDone. Refresh the PBIP / Direct Lake model to pick it up.')

## 4. Verify

Detects which shape landed and prints the relevant headline (surface mix, or budget/utilization).

In [ ]:
def _sum(df, c): return df.select(F.sum(F.coalesce(F.col(c).cast('double'), F.lit(0.0)))).collect()[0][0] or 0.0

t = spark.table(OUTPUT_TABLE); n = t.count()
users = t.select(F.countDistinct('User_Principal_Name')).collect()[0][0] if n else 0
print(f'{OUTPUT_TABLE}: {n:,} row(s), {users:,} distinct user(s)')
if n:
    cw, wq, ot = _sum(t,'Cowork_Credits'), _sum(t,'WorkIQ_Credits'), _sum(t,'Other_Credits')
    total, lim = _sum(t,'Total_Credits'), _sum(t,'Monthly_Credit_Limit')
    dated = t.filter(F.col('Last_Activity_Date').isNotNull()).count()
    print(f'  Total credits : {total:,.2f}')
    if cw or wq or ot:
        s = cw + wq + ot
        print(f'  surface mix — Cowork {cw/s:.1%} | WorkIQ {wq/s:.1%} | Other {ot/s:.1%}')
    if lim:
        print(f'  Total credit limit : {lim:,.0f}   utilization {total/lim:.1%}')
        print(f'  Total sessions     : {_sum(t,"Session_Count"):,.0f}')
    print(f'  Last_Activity_Date parsed (non-null): {dated:,}/{n:,}')
else:
    print('  (empty table — no export landed; pages stay dormant via Enable_CostConsumption.)')

---
**Connect the PBIP**: feeds the `copilot_cost_consumption` model query, gated by `Enable_CostConsumption`. `User_Principal_Name` joins org data (`Chat + Agent Org Data[PersonId_Normalized]`); `Last_Activity_Date` joins `Calendar[Date]`. Additive to the PPAC `credit_consumption_*` tables.